# IPN-13: detector click ở tốc độ khung gốc vs 64 frame

Câu hỏi: detector `index_click_cycle` / `two_finger_click_cycle` gần ngẫu nhiên (0,40–0,54 trên train) vì **tensor đã bị resample về 64 frame** làm méo chu kỳ click, hay vì tín hiệu closure vốn không phân biệt 1 và 2 click?

Hai chế độ cùng split train/validation, cùng detector:
- `t64`: tensor `T64` đã resample + grid 18 cấu hình cũ (đối chứng);
- `fullrate`: cache `variable_v1` giữ nguyên tốc độ khung + grid 12 cấu hình có `temporal_max_gap`, `two_finger_start_delta` tính theo frame thật.

Chỉ train/validation, không có đường nào đọc test. Ngưỡng chấp nhận đặt trước trong cấu hình: coverage ≥ 0,50 và accuracy trên phần được phủ ≥ 0,75 ở cả G01/G08 và G02/G09.

In [ ]:
from pathlib import Path
import datetime as dt
import hashlib
import importlib.util
import itertools
import json
import shutil
import subprocess
import sys
import uuid
import zipfile

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Không chạy trên Colab: bỏ qua mount Drive.')

# --- Chỉ sửa các biến trong khối này ---
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_REF = 'HuyenHoang'
REPO_COMMIT = None  # Khuyến nghị điền commit chứa temporal factor V3
EXPECTED_COUNTS = {'train': 2443, 'val': 589}
MIN_VALIDATION_COVERAGE_PER_PAIR = 0.25
SMOKE_TARGET_LIMIT = None  # Ví dụ 24; None = chạy full target classes
MIN_ACCURACY_ON_COVERED = 0.75  # ngưỡng chấp nhận detector đếm click (đặt trước khi xem kết quả)
MIN_COVERAGE_ACCEPT = 0.50

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:10]
WORK_ROOT = Path('/content') / f'ipn13_click_fullrate_{RUN_ID}'
REPO_DIR = WORK_ROOT / 'repo'
LOCAL_OUTPUT = WORK_ROOT / 'calibration_output'   # chứa hai thư mục con: t64 và fullrate
DRIVE_OUTPUT_ROOT = DRIVE_DATA_ROOT / 'outputs' / 'ipn13_click_fullrate'
DRIVE_RUN_ROOT = DRIVE_DATA_ROOT / 'runs' / 'ipn13_click_fullrate'
print({'run_id': RUN_ID, 'scope': 'train_validation_only', 'smoke_target_limit': SMOKE_TARGET_LIMIT})

## 1. Clone code (phải push nhánh trước)

In [ ]:
COMMANDS = []
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def run_argv(argv, cwd=None, log_path=None):
    command = [str(value) for value in argv]
    COMMANDS.append(command)
    print('$', ' '.join(command))
    log = open(log_path, 'a', encoding='utf-8') if log_path else None
    try:
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            if log:
                log.write(line)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if log:
            log.close()

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

run_argv(['git', 'clone', '--filter=blob:none', REPO_URL, REPO_DIR])
run_argv(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_COMMIT or REPO_REF])
run_argv(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT or 'FETCH_HEAD'])
REPO_COMMIT_RESOLVED = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, text=True, capture_output=True
).stdout.strip()
if REPO_COMMIT:
    assert REPO_COMMIT_RESOLVED == REPO_COMMIT, (REPO_COMMIT_RESOLVED, REPO_COMMIT)
if (REPO_DIR / 'requirements.txt').is_file():
    run_argv([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_DIR / 'requirements.txt'])
CALIBRATION_SCRIPT = REPO_DIR / 'scripts' / 'calibrate_ipn_temporal_factors.py'
FACTOR_MODULE = REPO_DIR / 'final' / 'code' / 'src_neurosymbolic' / 'predicates' / 'temporal_factors.py'
assert CALIBRATION_SCRIPT.is_file(), 'Thiếu calibration script: hãy commit và push code mới trước.'
assert FACTOR_MODULE.is_file(), 'Thiếu temporal_factors.py: hãy commit và push code mới trước.'
help_text = subprocess.run([sys.executable, str(CALIBRATION_SCRIPT), '--help'], check=True, text=True, capture_output=True).stdout
assert '--test' not in help_text.lower() and 'test_manifest' not in help_text.lower()
print('Repository revision:', REPO_COMMIT_RESOLVED)

## 2. Kiểm tra dữ liệu, tensor T64 và cache variable_v1

In [ ]:
TRAIN_FILE = DRIVE_DATA_ROOT / 'train.txt'
VAL_FILE = DRIVE_DATA_ROOT / 'val.txt'
TRAIN_MANIFEST = DRIVE_DATA_ROOT / 'train_manifest.csv'
VAL_MANIFEST = DRIVE_DATA_ROOT / 'val_manifest.csv'
for path in (TRAIN_FILE, VAL_FILE, TRAIN_MANIFEST, VAL_MANIFEST):
    assert path.is_file(), f'Thiếu input: {path}'

def read_split(path):
    rows = []
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            source, _, label = line.rsplit(maxsplit=2)
            rows.append((source, int(label)))
    return rows

splits = {'train': read_split(TRAIN_FILE), 'val': read_split(VAL_FILE)}
for name, rows in splits.items():
    assert len(rows) == EXPECTED_COUNTS[name], (name, len(rows), EXPECTED_COUNTS[name])
    assert {label for _, label in rows} == set(range(13)), (name, sorted({label for _, label in rows}))
train_sources = {source.replace(chr(92), '/') for source, _ in splits['train']}
val_sources = {source.replace(chr(92), '/') for source, _ in splits['val']}
assert not (train_sources & val_sources), 'Train/validation source overlap'
CACHE_ROOT = next((path for path in (
    DRIVE_DATA_ROOT / 'keypoint_tensor_cache',
    DRIVE_DATA_ROOT / 'data' / 'keypoint_tensor_cache',
) if path.is_dir()), None)
assert CACHE_ROOT is not None, 'Không thấy keypoint_tensor_cache trên Drive.'
KEYPOINT_ROOT = DRIVE_DATA_ROOT / 'keypoints'
spec = importlib.util.spec_from_file_location('ipn_temporal_calibrator', CALIBRATION_SCRIPT)
calibrator = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = calibrator
spec.loader.exec_module(calibrator)
target_labels = {label for pair in calibrator.TARGET_PAIRS for label in pair}
preflight = {}
for name, split_file, manifest_file in (('train', TRAIN_FILE, TRAIN_MANIFEST), ('val', VAL_FILE, VAL_MANIFEST)):
    metadata = calibrator.manifest_labels(manifest_file)
    dataset = calibrator.KeypointSequenceDataset(split_file, KEYPOINT_ROOT, name, cache_root=CACHE_ROOT)
    resolved, missing = [], []
    for source, label_id in dataset.samples:
        calibrator.validate_source_split(source, name)
        label, _ = calibrator.label_from_source(source, name, metadata, label_id)
        if label not in target_labels:
            continue
        path = calibrator.resolve_sample_input(dataset, source)
        if path.suffix.lower() != '.pt' or not path.is_file():
            missing.append({'source': source, 'resolved': str(path)})
        else:
            resolved.append(path)
    assert not missing, f'{name}: thiếu tensor cache cho target samples; ví dụ {missing[:10]}'
    preflight[name] = len(resolved)
print({'counts': {name: len(rows) for name, rows in splits.items()}, 'cache_root': str(CACHE_ROOT), 'target_cache_preflight': preflight})
VARIABLE_CACHE_ROOT = CACHE_ROOT
assert (VARIABLE_CACHE_ROOT / 'variable_v1').is_dir(), 'Chưa có cache variable_v1 (bước 3 của kế hoạch chuỗi biến độ dài).'
sys.path[:0] = [str(REPO_DIR / 'final' / 'code')]
from neural_only_experiments.variable_dataset import variable_cache_path
missing_var = []
for name, split_file in (('train', TRAIN_FILE), ('val', VAL_FILE)):
    for source, label in splits[name]:
        if not variable_cache_path(source, VARIABLE_CACHE_ROOT).is_file():
            missing_var.append(source)
assert not missing_var, f'Thiếu {len(missing_var)} file variable_v1; ví dụ {missing_var[:5]}'
print('variable_v1 đủ cho train + val')


## 3. Grid cho chế độ t64 (đối chứng)

In [ ]:
click_settings = [
    {'click_active_threshold': .50, 'click_inactive_threshold': .42, 'click_min_confidence': .30, 'temporal_max_gap': 2},
    {'click_active_threshold': .56, 'click_inactive_threshold': .46, 'click_min_confidence': .35, 'temporal_max_gap': 3},
    {'click_active_threshold': .62, 'click_inactive_threshold': .50, 'click_min_confidence': .45, 'temporal_max_gap': 4},
]
sync_settings = [
    {'two_finger_start_delta': 3, 'two_finger_overlap': .50},
    {'two_finger_start_delta': 5, 'two_finger_overlap': .35},
]
pinch_settings = [
    {'pinch_min_excursion': .08, 'pinch_min_valid_ratio': .50, 'pinch_dominance_ratio': 1.10},
    {'pinch_min_excursion': .12, 'pinch_min_valid_ratio': .60, 'pinch_dominance_ratio': 1.20},
    {'pinch_min_excursion': .16, 'pinch_min_valid_ratio': .70, 'pinch_dominance_ratio': 1.35},
]
GRID = [{**click, **sync, **pinch} for click, sync, pinch in itertools.product(click_settings, sync_settings, pinch_settings)]
GRID_PATH = WORK_ROOT / 'temporal_factor_grid.json'
GRID_PATH.write_text(json.dumps(GRID, indent=2, sort_keys=True), encoding='utf-8')
print('Grid configs:', len(GRID), '->', GRID_PATH)

## 4. Chạy calibration hai chế độ

In [ ]:
T64_OUT, FULL_OUT = LOCAL_OUTPUT / 't64', LOCAL_OUTPUT / 'fullrate'
common = ['--data-root', DRIVE_DATA_ROOT, '--keypoint-root', KEYPOINT_ROOT, '--cache-root', CACHE_ROOT,
          '--min-coverage', MIN_VALIDATION_COVERAGE_PER_PAIR]
limit = ['--limit', SMOKE_TARGET_LIMIT] if SMOKE_TARGET_LIMIT is not None else []
# Đối chứng: tensor 64 frame đã resample + grid 18 cấu hình cũ
run_argv([sys.executable, '-u', CALIBRATION_SCRIPT, *common, '--grid', GRID_PATH, '--output-dir', T64_OUT, *limit],
         log_path=WORK_ROOT / 'calibration_t64.log')
# Thí nghiệm: tốc độ khung gốc từ variable_v1 + grid full-rate (12 cấu hình, gap/delta tính theo frame thật)
run_argv([sys.executable, '-u', CALIBRATION_SCRIPT, *common, '--variable-cache-root', VARIABLE_CACHE_ROOT,
          '--output-dir', FULL_OUT, *limit], log_path=WORK_ROOT / 'calibration_fullrate.log')
for out in (T64_OUT, FULL_OUT):
    assert (out / 'calibration_report.json').is_file() and (out / 'selected_config.json').is_file()


## 5. So sánh validation và kết luận theo tiêu chí đặt trước

In [ ]:
import csv
from collections import Counter

reports = {name: json.loads((out / 'calibration_report.json').read_text(encoding='utf-8'))
           for name, out in (('t64', T64_OUT), ('fullrate', FULL_OUT))}
print(f"{'cặp':8} {'chế độ':9} {'support':>7} {'coverage':>9} {'acc|covered':>12} {'acc(abstain=sai)':>17}")
verdict_inputs = {}
for pair in ('G01/G08', 'G02/G09', 'G10/G11'):
    for name, report in reports.items():
        m = report['selected_validation_score']['per_pair'][pair]
        verdict_inputs[(pair, name)] = m
        print(f"{pair:8} {name:9} {m['support']:7d} {m['coverage']:9.3f} {m['accuracy_on_covered']:12.3f} {m['overall_accuracy_abstain_wrong']:17.3f}")
    print()

print('Phân bố số click detector đếm được (validation, config đã chọn), theo nhãn thật:')
for name, out in (('t64', T64_OUT), ('fullrate', FULL_OUT)):
    index = reports[name]['selected_config_index']
    counts = {}
    with (out / 'per_sample.csv').open(encoding='utf-8', newline='') as handle:
        for row in csv.DictReader(handle):
            if row['split'] != 'val' or int(row['config_index']) != index:
                continue
            evidence = json.loads(row['evidence'])
            if 'count' in evidence:
                counts.setdefault(row['true_label'], Counter())[evidence['count']] += 1
    print(f'--- {name} (config {index})')
    for label in ('G01', 'G08', 'G02', 'G09'):
        histogram = counts.get(label, Counter())
        print(f'  {label}: ' + ', '.join(f'{k} click: {v}' for k, v in sorted(histogram.items())))

print('\nTiêu chí (đặt trước): coverage >= %.2f và accuracy trên phần được phủ >= %.2f cho cả hai cặp click.'
      % (MIN_COVERAGE_ACCEPT, MIN_ACCURACY_ON_COVERED))
accepted = {}
for name in reports:
    ok = all(verdict_inputs[(pair, name)]['coverage'] >= MIN_COVERAGE_ACCEPT
             and verdict_inputs[(pair, name)]['accuracy_on_covered'] >= MIN_ACCURACY_ON_COVERED
             for pair in ('G01/G08', 'G02/G09'))
    accepted[name] = ok
    print(f'  {name}: {"ĐẠT" if ok else "KHÔNG ĐẠT"}')
if accepted['fullrate'] and not accepted['t64']:
    print('KẾT LUẬN: tốc độ khung gốc cứu được detector click; có thể xây pair-residual trên factor full-rate.')
elif accepted['fullrate']:
    print('KẾT LUẬN: detector đạt ở cả hai chế độ; ưu tiên full-rate nếu hơn rõ về accuracy hoặc coverage.')
else:
    print('KẾT LUẬN: detector click vẫn không đủ tốt kể cả ở tốc độ khung gốc; không tích hợp pair-residual.')


## 6. Đóng gói ZIP (từ chối ghi đè)

In [ ]:
import torch
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
drive_output = DRIVE_OUTPUT_ROOT / RUN_ID
run_zip = DRIVE_RUN_ROOT / f'{RUN_ID}.zip'
if drive_output.exists() or run_zip.exists():
    raise FileExistsError(f'Từ chối ghi đè: {drive_output} hoặc {run_zip}')
shutil.copytree(LOCAL_OUTPUT, drive_output)
shutil.copy2(GRID_PATH, drive_output / 'temporal_factor_grid_t64.json')
for log in ('calibration_t64.log', 'calibration_fullrate.log'):
    shutil.copy2(WORK_ROOT / log, drive_output / log)
(drive_output / 'notebook_manifest.json').write_text(json.dumps({
    'run_id': RUN_ID, 'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'scope': 'train_validation_only', 'repo_commit': REPO_COMMIT_RESOLVED, 'commands': COMMANDS,
    'inputs': {str(p): sha256_file(p) for p in (TRAIN_FILE, VAL_FILE, TRAIN_MANIFEST, VAL_MANIFEST)},
    'accepted': accepted, 'min_accuracy_on_covered': MIN_ACCURACY_ON_COVERED,
    'min_coverage_accept': MIN_COVERAGE_ACCEPT, 'smoke_target_limit': SMOKE_TARGET_LIMIT,
}, indent=2), encoding='utf-8')
with zipfile.ZipFile(run_zip, 'x', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(drive_output.rglob('*')):
        if path.is_file():
            archive.write(path, arcname=path.relative_to(drive_output))
print('ZIP:', run_zip, sha256_file(run_zip))
